In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

## Read and Write it as Delta table

In [ ]:
df = (
    spark.read.format('parquet')
    .load("abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/raw_data/invoices_101_200.parquet")
)

display(df.limit(2))


In [ ]:
table_path = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/processed_data/"

In [ ]:
(
    df.write.format('delta')
    .mode('overwrite')
    .save(table_path)
)

## History Of Delta table

##

In [ ]:
delta_df = DeltaTable.forPath(spark, table_path)

In [ ]:
display(delta_df.history().limit(5))

## Updating the Delta Lake Table

In [ ]:
source_df = spark.read.parquet('abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/raw_data/invoices_1_100.parquet')
target_df = DeltaTable.forPath(spark, table_path)

(
    target_df.alias("target").merge(
        source_df.alias("src"),
        "src.invoice_no == target.invoice_no AND src.customer_id == target.customer_id"
    ).whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [ ]:
display(delta_df.history().limit(5))

In [ ]:
# DELETE FROM deltacatalog.deltadb.invoices
# WHERE customer_id = 99;
delta_df.delete("customer_id = 99")

In [ ]:
display(delta_df.history().limit(5))

In [ ]:
display(delta_df.toDF().filter(F.col("customer_id")==99).limit(1))

## Insert Operation

In [ ]:
source_df = spark.read.parquet('abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/raw_data/invoices_201_99457.parquet')
target_df = DeltaTable.forPath(spark, table_path)

(
    target_df.alias("target").merge(
        source_df.alias("src"),
        "src.invoice_no == target.invoice_no AND src.customer_id == target.customer_id"
    ).whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [ ]:
delta_df.toDF().count()

In [ ]:
display(delta_df.history().limit(5))

## Delta Lake Architecture


<img src="https://github.com/afaqueahmad7117/databricks-masterclass/blob/main/delta_lake/docs/images/Delta%20Lake%20Architecture.png?raw=true" height=700/>

## Delta Table at Scale

Check how to delta tables handle large metadata information after millions of updates

In [ ]:
from tqdm.notebook import tqdm

source_df = spark.read.parquet('abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/raw_data/invoices_201_99457.parquet')

for i in tqdm(range(50)):
    (
        source_df.write.format('delta')
        .mode('append')
        .save(table_path)
    )


<img src="https://github.com/afaqueahmad7117/databricks-masterclass/blob/main/delta_lake/docs/images/Handling%20Massive%20Metadata.png?raw=true" height = 700/>

- Here the delta table will compact and put the json changes into one single checkpoint parquet after finding the summarization
- then when reading the delta table will only read from the latest checkpoint and consecutive json files beyond that


# Why ACID Matters — Delta Lake vs Plain Parquet/File Systems

Plain Parquet files sitting in a data lake have **no built-in guarantees** about correctness during changes. Delta Lake adds a transaction log on top specifically to provide ACID — here's what each letter actually protects against, since the bank account examples are the clearest way to see it.

## Atomicity — no more "half-done" writes

**The problem without it:** Say you're transferring $500 from savings to checking. You deduct $500 from savings, update it — then the system crashes *before* checking gets its $500. With plain files, that money is just gone. There's no "undo."

**How Delta fixes it:** Every operation (insert/update/delete/merge) is wrapped as one **atomic transaction** in the delta log. Either the whole thing commits, or none of it does. A crash mid-write leaves the table exactly as it was before — no partial states, ever.

## Consistency — the data never ends up in a broken state

**The problem without it:** Two people simultaneously try to spend $80 and $60 from a $100 balance. Both read $100 at the same time, both think they have enough, both succeed — now the account is $140 overdrawn even though there was never that much to spend.

**How Delta fixes it:** Transactions can enforce rules (like constraints — `NOT NULL`, `price > 0`, etc., which we used in the schema validation labs) and check the *current* state before committing, not a stale one. A transaction that would violate the rules gets rejected outright rather than silently corrupting the table.

## Isolation — nobody sees your half-finished changes

**The problem without it:** If someone else could see your data mid-update — while you've deducted from savings but haven't yet added to checking — they'd see a table that never actually existed in a valid state.

**How Delta fixes it:** Delta uses **Optimistic Concurrency Control** (the mechanism we went deep on):
- No locks — multiple transactions can read/write at the same time.
- Each transaction reads a **version number** along with the data.
- Before committing, it checks: "is the version still what I started with?"
  - Same version → nobody else changed anything → commit safely.
  - Different version → someone else committed first → **fail and retry** with fresh data.
- Readers only ever see the last fully **committed** version — never someone else's in-progress changes.

This is the key thing plain files can't do at all — a reader hitting a Parquet file mid-rewrite might read a genuinely broken, half-written file.

## Durability — once committed, it's permanent

**The problem without it:** Your paycheck gets deposited, balance updates in memory — then the server crashes before it's actually persisted to disk. The backup that gets restored doesn't have your paycheck. It's just lost, with no record it ever happened.

**How Delta fixes it:** Every transaction is written to the **transaction log first**, and only marked complete once the "commit" entry is actually written. If a crash happens *before* the commit entry is written, Delta knows on restart to roll that transaction back — so you never end up with a "sort of, partially" saved change. Once the log says committed, it's committed for good.

---

## Why plain Parquet/data lakes can't do any of this
Parquet files are just files — there's no log tracking what changed, no version numbers to check against, and no built-in "all or nothing" wrapper around a set of writes. Any of these four failure modes (partial write, conflicting concurrent writes, readers seeing broken intermediate states, lost updates on crash) can happen with raw files, silently. Delta's transaction log (`_delta_log`) is the single mechanism that gives you all four guarantees at once — which is really the foundational reason Delta Lake exists over plain files in the first place.
